# Radical-Aligned Structure — One-Shot Reviewer-Proof Run

Free Colab T4. Hit **Run All** and walk away. Total ~2h 20m.

After every major step the cache is mirrored to Google Drive at
`MyDrive/radical_llm_v3/`. If the runtime dies mid-run, you can re-open
this notebook on a fresh runtime tomorrow and Run All again — every cell
checks the Drive checkpoint first and skips work that's already done.

The final cell zips everything and triggers a browser download.


In [ ]:
# 1. Clone repo (latest main with the perf knobs)
!rm -rf /content/repo
!git clone https://github.com/aryan35790jp/chinese_llm.git /content/repo
%cd /content/repo


In [ ]:
# 2. Install deps. The dependency-resolver warnings about numpy<2 are SAFE to ignore.
!pip install -q transformers==4.44.2 tokenizers==0.19.1 \
    numpy==1.26.4 pandas==2.2.2 scipy==1.13.1 scikit-learn==1.5.1 \
    matplotlib==3.9.0 tqdm==4.66.4 datasets==2.20.0 sentencepiece==0.2.0 \
    Pillow==10.4.0 umap-learn==0.5.6 \
    fugashi==1.3.2 unidic-lite==1.0.8 ipadic==1.0.0


In [ ]:
# 3. Mount Drive + define a checkpoint helper that we call after every step.
import os, subprocess, pathlib
from google.colab import drive
drive.mount('/content/drive', force_remount=False)

CKPT_ROOT = pathlib.Path('/content/drive/MyDrive/radical_llm_v3')
CKPT_ROOT.mkdir(parents=True, exist_ok=True)

def checkpoint(label):
    """Mirror the live runtime state to Drive. Call after every major step."""
    print(f'\n[ckpt] saving "{label}" to Drive ...')
    for sub in ['cache', 'results', 'data', 'figures']:
        src = pathlib.Path('/content/repo') / sub
        if src.exists() and any(src.iterdir()):
            dst = CKPT_ROOT / sub
            subprocess.run(['rsync', '-a', '--delete-after',
                            f'{src}/', f'{dst}/'], check=False)
    (CKPT_ROOT / 'last_step.txt').write_text(label)
    print(f'[ckpt] done — Drive contents:')
    !du -sh {CKPT_ROOT}/* 2>/dev/null || true

def restore_if_present(sub):
    """If a previous run saved this folder to Drive, copy it back to runtime."""
    src = CKPT_ROOT / sub
    dst = pathlib.Path('/content/repo') / sub
    if src.exists() and any(src.iterdir()):
        print(f'[restore] pulling {sub}/ from Drive ...')
        dst.mkdir(parents=True, exist_ok=True)
        subprocess.run(['rsync', '-a', f'{src}/', f'{dst}/'], check=False)
        return True
    return False

# Try to restore everything from any previous run on this Drive.
for sub in ['cache', 'results', 'data', 'figures']:
    restore_if_present(sub)

import torch
print('CUDA:', torch.cuda.is_available(),
      '|', torch.cuda.get_device_name(0) if torch.cuda.is_available() else '')


In [ ]:
# 4. Build dataset (Unihan + tokenizer filter, ~3 min). Skips if already done.
import pathlib
if not pathlib.Path('data/radical_dataset.csv').exists():
    !mkdir -p data/unihan
    !curl -sL -o data/unihan/Unihan.zip https://www.unicode.org/Public/UCD/latest/ucd/Unihan.zip
    !cd data/unihan && unzip -o -q Unihan.zip && cd ../..
    !python scripts/new/dataset_builder.py
    !python scripts/new/classify_liushu.py
    checkpoint('dataset_built')
else:
    print('dataset already built, skipping')


In [ ]:
# 5. Tokenization audit (1 min)
import os, pathlib
os.environ['RADICAL_FAST'] = '1'
if not pathlib.Path('results/tokenization_audit_summary.csv').exists():
    !python scripts/new/tokenization_audit.py
    checkpoint('audit_done')
else:
    print('audit already done, skipping')


In [ ]:
# 6. THE HEAVY STEP — extract embeddings for 10 models (~50 min on T4).
#    Per-model caching means if this dies partway, the next run resumes.
import os, pathlib
os.environ['RADICAL_FAST'] = '1'
!RADICAL_FAST=1 python scripts/new/extract_embeddings.py
checkpoint('embeddings_extracted')


In [ ]:
# 7. Pure-vision baseline (~3 min)
import pathlib
if not pathlib.Path('cache/embeddings/glyph_only__resnet18').exists():
    !apt-get install -y -q fonts-noto-cjk 2>/dev/null || true
    !python scripts/new/glyph_only_baseline.py
    checkpoint('glyph_baseline_done')
else:
    print('glyph baseline cached, skipping')


In [ ]:
# 8. Isotropy correction (~3 min)
!python scripts/new/isotropy_correction.py
checkpoint('isotropy_done')


In [ ]:
# 9. Layer-wise cohesion (~7 min, 10 models)
import os
os.environ['RADICAL_FAST'] = '1'
!RADICAL_FAST=1 python scripts/new/layer_wise_analysis.py
checkpoint('layerwise_done')


In [ ]:
# 10. Pseudoradical specificity test (~18 min with B=100).
#     Cuts permutation count from 200 to 100; p-floor still ≤ 0.0099.
import os
os.environ['RADICAL_FAST'] = '1'
os.environ['RADICAL_PSEUDO_B'] = '100'
!RADICAL_FAST=1 RADICAL_PSEUDO_B=100 python scripts/new/pseudoradical_control.py
checkpoint('pseudoradical_done')


In [ ]:
# 11. Frequency-matched + random-init + semantic + phon-vs-sem + cross-script + glyph-comp + scaling + orth-arith
#     All small (~7 min combined)
import os
os.environ['RADICAL_FAST'] = '1'
!RADICAL_FAST=1 python scripts/new/frequency_matched_pairs.py
!RADICAL_FAST=1 python scripts/new/random_init_baseline.py
!RADICAL_FAST=1 python scripts/new/expanded_semantic_control.py
!RADICAL_FAST=1 python scripts/new/phonetic_vs_semantic_radicals.py
!RADICAL_FAST=1 python scripts/new/cross_script_japanese.py
!RADICAL_FAST=1 python scripts/new/glyph_comparison.py
!RADICAL_FAST=1 python scripts/new/scaling_analysis.py
!RADICAL_FAST=1 python scripts/new/orthographic_arithmetic.py
!RADICAL_FAST=1 python scripts/new/downstream_validation.py
checkpoint('cheap_analyses_done')


In [ ]:
# 12. Variance decomposition (~25 min — Wikipedia stream first time, then cached)
import os
os.environ['RADICAL_FAST'] = '1'
!RADICAL_FAST=1 python scripts/new/cooccurrence_baseline.py
checkpoint('variance_decomp_done')


In [ ]:
# 13. Cluster-robust SE regression (~3 min) + non-parametric stats (~3 min)
!python scripts/new/cluster_robust_regression.py
!python scripts/new/nonparametric_stats.py
checkpoint('robust_stats_done')


In [ ]:
# 14. Procedural cloze items + probe (~25 min for 7 models, the behavioural evidence)
import pathlib
for f in ['radical_cloze.csv', 'radical_cloze_summary.csv']:
    p = pathlib.Path('results') / f
    if p.exists(): p.unlink()
!python scripts/new/cloze_procedural.py
!python scripts/new/radical_cloze_probe.py
checkpoint('cloze_done')


In [ ]:
# 15. Figures + auto report
!python scripts/new/figures.py
!python scripts/new/results_report.py
checkpoint('figures_and_report_done')

with open('results/_REPORT.md', encoding='utf-8') as f:
    from IPython.display import Markdown
    md_text = f.read()
Markdown(md_text)


In [ ]:
# 16. PACK AND DOWNLOAD
!cd /content/repo && zip -qr /content/results_v3.zip results figures data/radical_dataset.csv data/radical_summary.csv
!ls -lh /content/results_v3.zip
from google.colab import files
files.download('/content/results_v3.zip')
print('\n=== ALL DONE — zip downloading to your browser ===')
print('Drive checkpoint also lives at MyDrive/radical_llm_v3/')
